In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "bank-full.csv",
    sep=";"
)

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
df.shape

(45211, 17)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 8.1 MB


In [6]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

In [7]:
numerical_columns = [
    "age",
    "balance",
    "day",
    "duration",
    "campaign",
    "pdays",
    "previous"
]

categorical_columns = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome"
]

target_column = "y"

In [8]:
for column in categorical_columns:
    print("\n")
    print(f"COLUMN: {column}")
    print(df[column].unique())



COLUMN: job
<ArrowStringArray>
[   'management',    'technician',  'entrepreneur',   'blue-collar',
       'unknown',       'retired',        'admin.',      'services',
 'self-employed',    'unemployed',     'housemaid',       'student']
Length: 12, dtype: str


COLUMN: marital
<ArrowStringArray>
['married', 'single', 'divorced']
Length: 3, dtype: str


COLUMN: education
<ArrowStringArray>
['tertiary', 'secondary', 'unknown', 'primary']
Length: 4, dtype: str


COLUMN: default
<ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str


COLUMN: housing
<ArrowStringArray>
['yes', 'no']
Length: 2, dtype: str


COLUMN: loan
<ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str


COLUMN: contact
<ArrowStringArray>
['unknown', 'cellular', 'telephone']
Length: 3, dtype: str


COLUMN: month
<ArrowStringArray>
['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan', 'feb', 'mar', 'apr',
 'sep']
Length: 12, dtype: str


COLUMN: poutcome
<ArrowStringArray>
['unknown', 'failure', 'other', 'success'

In [9]:
df["y"].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [10]:
conversion_rate = (
    df["y"]
    .value_counts(normalize=True) * 100
)

conversion_rate

y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64

In [11]:
job_conversion = (
    df.groupby("job")["y"]
    .value_counts(normalize=True)
    .unstack() * 100
)

job_conversion

y,no,yes
job,,
admin.,87.797331,12.202669
blue-collar,92.725031,7.274969
entrepreneur,91.728312,8.271688
housemaid,91.209677,8.790323
management,86.244449,13.755551
retired,77.208481,22.791519
self-employed,88.157061,11.842939
services,91.116996,8.883004
student,71.321962,28.678038


In [12]:
job_conversion_sorted = (
    job_conversion
    .sort_values(by="yes", ascending=False)
)

job_conversion_sorted

y,no,yes
job,,
student,71.321962,28.678038
retired,77.208481,22.791519
unemployed,84.497314,15.502686
management,86.244449,13.755551
admin.,87.797331,12.202669
self-employed,88.157061,11.842939
unknown,88.194444,11.805556
technician,88.943004,11.056996
services,91.116996,8.883004


In [13]:
job_summary = (
    df.groupby("job")
    .agg(
        total_customers=("job", "count"),
        avg_balance=("balance", "mean"),
        avg_call_duration=("duration", "mean")
    )
)

job_summary

,total_customers,avg_balance,avg_call_duration
job,,,
admin.,5171,1135.838909,246.896732
blue-collar,9732,1078.826654,262.901562
entrepreneur,1487,1521.470074,256.309348
housemaid,1240,1392.395161,245.825000
management,9458,1763.616832,253.995771
retired,2264,1984.215106,287.361307
self-employed,1579,1647.970868,268.157061
services,4154,997.088108,259.318729
student,938,1388.060768,246.656716


In [14]:
job_analysis = job_summary.merge(
    job_conversion_sorted,
    on="job"
)

job_analysis

,total_customers,avg_balance,avg_call_duration,no,yes
job,,,,,
admin.,5171,1135.838909,246.896732,87.797331,12.202669
blue-collar,9732,1078.826654,262.901562,92.725031,7.274969
entrepreneur,1487,1521.470074,256.309348,91.728312,8.271688
housemaid,1240,1392.395161,245.825000,91.209677,8.790323
management,9458,1763.616832,253.995771,86.244449,13.755551
retired,2264,1984.215106,287.361307,77.208481,22.791519
self-employed,1579,1647.970868,268.157061,88.157061,11.842939
services,4154,997.088108,259.318729,91.116996,8.883004
student,938,1388.060768,246.656716,71.321962,28.678038


In [15]:
job_analysis = job_analysis.round(2)

job_analysis

,total_customers,avg_balance,avg_call_duration,no,yes
job,,,,,
admin.,5171,1135.84,246.90,87.80,12.20
blue-collar,9732,1078.83,262.90,92.73,7.27
entrepreneur,1487,1521.47,256.31,91.73,8.27
housemaid,1240,1392.40,245.82,91.21,8.79
management,9458,1763.62,254.00,86.24,13.76
retired,2264,1984.22,287.36,77.21,22.79
self-employed,1579,1647.97,268.16,88.16,11.84
services,4154,997.09,259.32,91.12,8.88
student,938,1388.06,246.66,71.32,28.68


In [16]:
top_segments = (
    job_analysis
    .sort_values(by="yes", ascending=False)
    .head(5)
)

top_segments

,total_customers,avg_balance,avg_call_duration,no,yes
job,,,,,
student,938,1388.06,246.66,71.32,28.68
retired,2264,1984.22,287.36,77.21,22.79
unemployed,1303,1521.75,288.54,84.50,15.50
management,9458,1763.62,254.00,86.24,13.76
admin.,5171,1135.84,246.90,87.80,12.20


In [17]:
best_segment = top_segments.index[0]

best_conversion = top_segments["yes"].iloc[0]

print(
    f"The highest converting customer segment is "
    f"{best_segment} with a conversion rate of "
    f"{best_conversion:.2f}%."
)

The highest converting customer segment is student with a conversion rate of 28.68%.


In [18]:
worst_segment = (
    job_analysis
    .sort_values(by="yes")
    .head(1)
)

worst_segment

,total_customers,avg_balance,avg_call_duration,no,yes
job,,,,,
blue-collar,9732,1078.83,262.9,92.73,7.27


In [19]:
for column in categorical_columns:
    
    unknown_count = (
        df[column] == "unknown"
    ).sum()
    
    print(
        f"{column}: {unknown_count}"
    )

job: 288
marital: 0
education: 1857
default: 0
housing: 0
loan: 0
contact: 13020
month: 0
poutcome: 36959


In [20]:
education_analysis = (
    df.groupby("education")["y"]
    .value_counts(normalize=True)
    .unstack() * 100
)

education_analysis.round(2)

y,no,yes
education,,
primary,91.37,8.63
secondary,89.44,10.56
tertiary,84.99,15.01
unknown,86.43,13.57


In [21]:
def conversion_analysis(column_name):
    
    analysis = (
        df.groupby(column_name)["y"]
        .value_counts(normalize=True)
        .unstack() * 100
    )
    
    analysis = analysis.round(2)
    
    return analysis.sort_values(
        by="yes",
        ascending=False
    )

In [22]:
conversion_analysis("job")

y,no,yes
job,,
student,71.32,28.68
retired,77.21,22.79
unemployed,84.50,15.50
management,86.24,13.76
admin.,87.80,12.20
self-employed,88.16,11.84
unknown,88.19,11.81
technician,88.94,11.06
services,91.12,8.88


In [23]:
conversion_analysis("education")

y,no,yes
education,,
tertiary,84.99,15.01
unknown,86.43,13.57
secondary,89.44,10.56
primary,91.37,8.63


In [24]:
conversion_analysis("marital")
conversion_analysis("loan")

y,no,yes
loan,,
no,87.34,12.66
yes,93.32,6.68


In [25]:
numerical_analysis = (
    df.groupby("y")[numerical_columns]
    .mean()
)

numerical_analysis.round(2)

,age,balance,day,duration,campaign,pdays,previous
y,,,,,,,
no,40.84,1303.71,15.89,221.18,2.85,36.42,0.50
yes,41.67,1804.27,15.16,537.29,2.14,68.70,1.17


In [26]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

In [27]:
df.head(20)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
5,35,management,married,tertiary,no,231,yes,no,unknown,5,may,139,1,-1,0,unknown,no
6,28,management,single,tertiary,no,447,yes,yes,unknown,5,may,217,1,-1,0,unknown,no
7,42,entrepreneur,divorced,tertiary,yes,2,yes,no,unknown,5,may,380,1,-1,0,unknown,no
8,58,retired,married,primary,no,121,yes,no,unknown,5,may,50,1,-1,0,unknown,no
9,43,technician,single,secondary,no,593,yes,no,unknown,5,may,55,1,-1,0,unknown,no


In [29]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [30]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 25, 35, 45, 60, 100],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-60",
        "60+"
    ]
)

In [31]:
df[["age", "age_group"]].head(20)

,age,age_group
0,58,46-60
1,44,36-45
2,33,26-35
3,47,46-60
4,33,26-35
5,35,26-35
6,28,26-35
7,42,36-45
8,58,46-60
9,43,36-45


In [32]:
conversion_analysis("age_group")

y,no,yes
age_group,,
60+,57.74,42.26
18-25,76.36,23.64
26-35,88.00,12.00
46-60,90.22,9.78
36-45,90.61,9.39


In [33]:
df["balance_category"] = pd.cut(
    df["balance"],
    bins=[-10000, 0, 1000, 5000, 100000],
    labels=[
        "Negative",
        "Low Balance",
        "Medium Balance",
        "High Balance"
    ]
)

In [34]:
conversion_analysis("balance_category")

y,no,yes
balance_category,,
High Balance,84.49,15.51
Medium Balance,84.67,15.33
Low Balance,89.10,10.90
Negative,93.10,6.90


In [35]:
df["campaign_intensity"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 3, 5, 100],
    labels=[
        "1 Contact",
        "2-3 Contacts",
        "4-5 Contacts",
        "Heavy Contact"
    ]
)

In [36]:
conversion_analysis("campaign_intensity")

y,no,yes
campaign_intensity,,
1 Contact,85.40,14.60
2-3 Contacts,88.80,11.20
4-5 Contacts,91.37,8.63
Heavy Contact,94.19,5.81


In [37]:
executive_kpis = {
    
    "Total Customers":
        len(df),
    
    "Successful Conversions":
        (df["y"] == "yes").sum(),
    
    "Conversion Rate":
        round(
            (df["y"] == "yes").mean() * 100,
            2
        ),
    
    "Average Balance":
        round(
            df["balance"].mean(),
            2
        ),
    
    "Average Call Duration":
        round(
            df["duration"].mean(),
            2
        ),
    
    "Average Campaign Calls":
        round(
            df["campaign"].mean(),
            2
        )
}

In [38]:
kpi_table = pd.DataFrame(
    executive_kpis.items(),
    columns=["KPI", "Value"]
)

kpi_table

,KPI,Value
0,Total Customers,45211.00
1,Successful Conversions,5289.00
2,Conversion Rate,11.70
3,Average Balance,1362.27
4,Average Call Duration,258.16
5,Average Campaign Calls,2.76


In [39]:
segment_kpis = (
    df.groupby("age_group")
    .agg(
        total_customers=("age_group", "count"),
        avg_balance=("balance", "mean"),
        avg_duration=("duration", "mean"),
        conversion_rate=(
            "y",
            lambda x: (
                (x == "yes").mean() * 100
            )
        )
    )
)

segment_kpis.round(2)

,total_customers,avg_balance,avg_duration,conversion_rate
age_group,,,,
18-25,1324,902.73,272.30,23.64
26-35,15571,1147.20,263.43,12.00
36-45,13856,1319.58,255.57,9.39
46-60,13260,1588.30,248.12,9.78
60+,1188,2678.45,316.08,42.26


In [40]:
analytics_df = df.copy()

In [44]:
analytics_df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y', 'age_group', 'balance_category',
       'campaign_intensity'],
      dtype='str')

In [45]:
analytics_df.to_csv(
    "bank_analytics_clean.csv",
    index=False
)